# B-cell Epitope Analysis

Parameterised template — run via papermill, do not execute directly.

```bash
uv run papermill notebooks/epitope_analysis.ipynb outputs/ERCC1/epitope_analysis.ipynb -p TARGET ERCC1
```

In [ ]:
# papermill injects TARGET when executing
TARGET = "ERCC1"

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

# Locate repo root by walking up until pyproject.toml is found
_cwd = Path().resolve()
REPO_ROOT = next(p for p in [_cwd, *_cwd.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from epitope_pipeline.integration.combine import load_and_combine

target_dir = REPO_ROOT / "data" / TARGET
out_dir    = REPO_ROOT / "outputs" / TARGET
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Target : {TARGET}")
print(f"Data   : {target_dir}")
print(f"Outputs: {out_dir}")

In [ ]:
df = load_and_combine(target_dir)
print('Sequence length:', len(df))
df.head()

In [ ]:
df.to_csv(out_dir / 'combined_scores.csv', index=False)
print('Saved:', out_dir / 'combined_scores.csv')

for col in ['is_epitope_bepipred', 'is_epitope_discotope', 'is_epitope_AND']:
    n = df[col].sum()
    print(f'  {col}: {n} / {len(df)} ({100*n/len(df):.1f}%)')

In [ ]:
sns.set_style('whitegrid')
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df['res_id'], df['bepipred_score'],
        color='mediumpurple', linewidth=2, label='Epitope Score')
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Threshold (0.50)')
ax.fill_between(df['res_id'], 0, df['bepipred_score'],
                where=df['is_epitope_bepipred'],
                alpha=0.8, color='lavender', label='Predicted Linear Epitope')

ax.set_xlabel('Residue Position', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'BepiPred 3.0: Linear B-cell Epitope Prediction — {TARGET}', fontsize=14, fontweight='bold')
ax.set_ylim(-0.05, max(0.6, df['bepipred_score'].max() * 1.1))
ax.legend()
plt.tight_layout()
fig.savefig(out_dir / 'bepipred_profile.png', dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df['res_id'], df['discotope_score'],
        color='green', linewidth=2, label='Epitope Score')
ax.axhline(y=0.9, color='gray', linestyle='--', linewidth=1, label='Threshold (0.90)')
ax.fill_between(df['res_id'], 0, df['discotope_score'],
                where=df['is_epitope_discotope'],
                alpha=0.4, color='mediumseagreen', label='Predicted Conformational Epitope')

ax.set_xlabel('Residue Position', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'DiscoTope 3.0: Conformational B-cell Epitope Prediction — {TARGET}', fontsize=14, fontweight='bold')
ax.set_ylim(-0.1, max(3.5, df['discotope_score'].max() * 1.1))
ax.legend()
plt.tight_layout()
fig.savefig(out_dir / 'discotope_profile.png', dpi=300)
plt.show()

In [ ]:
sns.set_style('whitegrid')
fig = plt.figure(figsize=(18, 8))
gs  = fig.add_gridspec(2, 1, hspace=0.4)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

ax1.plot(df['res_id'], df['bepipred_score'], color='mediumpurple', linewidth=2, label='Score')
ax1.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Threshold (0.50)')
ax1.fill_between(df['res_id'], 0, df['bepipred_score'],
                 where=df['is_epitope_bepipred'], alpha=0.8, color='lavender', label='Epitope')
ax1.set_ylabel('BepiPred Score', fontsize=11, fontweight='bold')
ax1.set_title(f'Linear B-cell Epitopes (BepiPred 3.0) — {TARGET}', fontsize=13, fontweight='bold')
ax1.set_ylim(-0.05, max(0.65, df['bepipred_score'].max() * 1.1))
ax1.legend()

ax2.plot(df['res_id'], df['discotope_score'], color='green', linewidth=2, label='Score')
ax2.axhline(y=0.9, color='gray', linestyle='--', linewidth=1, label='Threshold (0.90)')
ax2.fill_between(df['res_id'], 0, df['discotope_score'],
                 where=df['is_epitope_discotope'], alpha=0.4, color='mediumseagreen', label='Epitope')
ax2.set_ylabel('DiscoTope Score', fontsize=11, fontweight='bold')
ax2.set_title(f'Conformational B-cell Epitopes (DiscoTope 3.0) — {TARGET}', fontsize=13, fontweight='bold')
ax2.set_xlabel('Residue Position', fontsize=11, fontweight='bold')
ax2.legend()

fig.savefig(out_dir / 'B_cell_epitope_combined.png', dpi=300, bbox_inches='tight')
print('Saved:', out_dir / 'B_cell_epitope_combined.png')
plt.show()

In [ ]:
epitopes = df[df['is_epitope_AND']]
print(f'{len(epitopes)} epitope residues (is_epitope_AND) out of {len(df)}')
display(epitopes)